# AutoLeitura — Treinamento no Google Colab

Este notebook treina os dois modelos do pipeline na GPU gratuita do Colab:

1. **YOLOv8** → detecta a região do contador
2. **CRNN + CTC** → lê os dígitos do recorte

No final, os pesos (`detector.pt`, `recognizer.pt`) e o banco com as métricas são salvos no Google Drive.
A **interface web roda no notebook do grupo** (`streamlit run app.py`), usando esses pesos.

> Antes de começar: `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.

In [ ]:
!nvidia-smi -L

## 1. Projeto
O código é baixado direto do GitHub. O Drive serve para guardar os datasets e os pesos treinados
(eles se perdem quando a sessão do Colab acaba).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/AutoLeitura'
!mkdir -p $DRIVE
!rm -rf /content/AutoLeitura && git clone -q https://github.com/joaopferreira7/AutoLeitura.git /content/AutoLeitura
%cd /content/AutoLeitura
!pip install -q -r requirements.txt

## 2. Dados
Escolha **uma** opção (pode combinar A + B com `--append` para ter energia **e** água).

**A) UFPR-AMR (principal).** Cole o link de download que os autores enviaram por e-mail.
O link é pessoal e a licença é só para uso acadêmico: **não escreva o link no notebook**, porque o repositório é público.
O zip (~5 GB) é baixado direto do servidor da UFPR para o Colab, sem passar pelo seu computador.
Com `GUARDAR_NO_DRIVE = True`, uma cópia fica no Drive e as próximas sessões não precisam baixar de novo.

In [ ]:
import os
from getpass import getpass

GUARDAR_NO_DRIVE = False                       # True guarda o zip (~5 GB) no Drive para as próximas sessões
ZIP_DRIVE = f'{DRIVE}/UFPR-AMR.zip'
ZIP = ZIP_DRIVE if (GUARDAR_NO_DRIVE or os.path.exists(ZIP_DRIVE)) else '/content/UFPR-AMR.zip'

if not os.path.exists(ZIP):
    url = getpass('Link do UFPR-AMR (do e-mail dos autores): ').strip()
    !wget -q --show-progress -O "$ZIP" "$url"

!mkdir -p data/raw/ufpr && unzip -oq "$ZIP" -d data/raw/ufpr/
!find data/raw/ufpr -maxdepth 2 -type d
!python scripts/prepare_data.py --source ufpr --input data/raw/ufpr

**B) Roboflow Universe (medidores de água).** Escolha um dataset com classes de dígito `0`–`9`
(um box por dígito), clique em *Download Dataset → YOLOv8 → show download code* e copie o workspace/projeto/versão.

In [ ]:
!pip install -q roboflow
from getpass import getpass
from roboflow import Roboflow
rf = Roboflow(api_key=getpass('Roboflow API key: '))
ds = rf.workspace('WORKSPACE').project('PROJETO').version(1).download('yolov8', location='data/raw/agua')
!python scripts/prepare_data.py --source roboflow --input data/raw/agua --append

**C) Sintético (só para testar o pipeline, não usar como resultado).**

In [ ]:
!python scripts/make_synthetic.py --n 2000 --out data/raw/synthetic
!python scripts/prepare_data.py --source ufpr --input data/raw/synthetic

## 3. Treino do detector (YOLOv8)

In [ ]:
!python train_detector.py --model yolov8s.pt --epochs 100 --batch 32 --device 0

In [ ]:
from IPython.display import Image, display
display(Image('runs/detector/train/results.png', width=900))
display(Image('runs/detector/train/val_batch0_pred.jpg', width=900))

## 4. Treino do reconhecedor (CRNN)

In [ ]:
!python train_recognizer.py --epochs 80 --batch 64

## 4b. Treino do leitor de dígitos (YOLOv8 com 10 classes)
Segundo leitor: detecta e classifica cada dígito no recorte. O pipeline usa a leitura dele e a CRNN como conferência.

In [ ]:
!python train_digits.py --epochs 150 --device 0

## 5. Avaliação no conjunto de teste

In [ ]:
!python evaluate.py --errors erros_teste.csv

In [ ]:
import pandas as pd
pd.read_csv('erros_teste.csv').head(20)

## 5b. Medidores de água (opcional)
Dataset de hidrômetros word-wheel (Zhao et al., Dryad, 2026, licença CC0): https://doi.org/10.5061/dryad.7d7wm3860

O Dryad bloqueia downloads por script: baixe `Word-Wheel_Water_Meter_Dataset.zip` (2,6 GB) pelo navegador,
envie para `MeuDrive/AutoLeitura/` e rode as células abaixo. Na água, o modelo de detecção é de **segmentação**
da janela dos dígitos (ela é endireitada antes da leitura) e duas CRNNs leem nos dois sentidos (a segunda dá a segunda opinião).

In [ ]:
!mkdir -p data/raw/agua && unzip -oq $DRIVE/Word-Wheel_Water_Meter_Dataset.zip -d data/raw/agua/
!python scripts/prepare_agua.py --input data/raw/agua/Word-Wheel_Water_Meter_Dataset
!python train_detector.py --tipo agua --epochs 25 --fracao 0.15 --device 0
!python train_recognizer.py --tipo agua --epochs 40 --peso-6 3 --semente 2
!python train_recognizer.py --tipo agua --epochs 40 --saida weights/recognizer_agua_2.pt
!python evaluate.py --tipo agua

## 6. Salvar no Drive

In [ ]:
!mkdir -p $DRIVE/weights $DRIVE/runs
!cp weights/*.pt $DRIVE/weights/
!cp autoleitura.db erros_teste.csv $DRIVE/
!cp -r runs/detector $DRIVE/runs/
!ls -la $DRIVE/weights

## 7. Testar uma foto aqui mesmo

In [ ]:
from google.colab import files
import cv2
from autoleitura.pipeline import AutoLeitura, draw_result
from autoleitura.preprocess import load_image

system = AutoLeitura()
for name, data in files.upload().items():
    img = load_image(data)
    res = system.read(img)
    print(name, '->', res['reading'], f"({res['confidence']*100:.1f}%)", res['warning'] or '')
    cv2.imwrite('resultado.jpg', draw_result(img, res))
    display(Image('resultado.jpg', width=700))